<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/05_serve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · 5. Serve with vLLM

The production path: vLLM serves Mistral-7B with FP8 weights and the FinSight adapter, and a small FastAPI app
in front of it takes a release and returns the JSON with timing. Use a **fresh** runtime: vLLM installs its own
torch. This path was not run in the repo's CI (no GPU), so the first run may need small fixes; if vLLM will not
start on a T4, the last cell serves the same API without it.

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"   # to try a branch before merging it, put its name here
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements-serve.txt

In [ ]:
# Keep data, checkpoints and results on Google Drive so a disconnect doesn't lose them
# and every notebook sees the same dataset.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!mkdir -p /content/drive/MyDrive/finsight/data
!rm -rf outputs results data && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results && ln -s /content/drive/MyDrive/finsight/data data

In [ ]:
# Mistral-7B-Instruct-v0.3 is gated: accept its terms at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3, then paste a read token here
# (or store it as the Colab secret HF_TOKEN).
from huggingface_hub import login
login()

In [ ]:
# The SEC asks every client to identify itself with a name and email (sent only to sec.gov).
# Store it once as a Colab secret named SEC_USER_AGENT, or type it when asked.
import os
try:
    from google.colab import userdata
    os.environ["SEC_USER_AGENT"] = userdata.get("SEC_USER_AGENT")
except Exception:
    os.environ["SEC_USER_AGENT"] = input("Your name and email for the SEC, e.g. Jane Doe jane@example.com: ")

In [ ]:
import subprocess, time, urllib.request
vllm = subprocess.Popen(["bash", "scripts/serve_vllm.sh", "outputs/finsight-earnings/final"],
                        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health"); print("vLLM is up"); break
    except Exception:
        time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
api = subprocess.Popen(["python", "-m", "finsight.serve", "--backend", "vllm"], stdout=open("api.log", "w"),
                       stderr=subprocess.STDOUT)
time.sleep(5)
import json
release = json.loads(open("data/earnings/test_earnings.jsonl").readline())
req = urllib.request.Request("http://localhost:8080/v1/extract", data=json.dumps({"url": release["url"]}).encode(),
                             headers={"content-type": "application/json"})
print(json.load(urllib.request.urlopen(req)))
print("gold:", release["target"])

## Record the CI baseline
Runs 50 test releases through the served model and writes `results/ci_baseline.json` and `results/ci_sample.jsonl`; commit both to arm the CI smoke eval.

In [ ]:
!python -m finsight.ci_eval --endpoint http://localhost:8000 --write-baseline

In [ ]:
!python -m finsight.monitor logs/requests.jsonl

## Fallback without vLLM

In [ ]:
# !pip install -q -r requirements.txt fastapi uvicorn && python -m finsight.serve --backend transformers